# Qual é o comprimento de uma linha costeira? Dois estimadores e uma lição de análise de dados

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fsaliby/Coastline_Length_Sampling/blob/main/notebooks/coastline_notebook_pt.ipynb)

Este notebook mede o comprimento de uma linha costeira de duas maneiras e usa os
resultados para introduzir ideias que valem para qualquer experimento: estimadores e
suas distribuições amostrais, ajuste de leis de potência, propagação de incerteza por
Monte Carlo, incerteza estatística versus sistemática e validação com respostas
conhecidas.

- **Pontos aleatórios (RP).** Sorteamos $n_p$ pontos ao longo da costa, ligamos esses
  pontos em ordem com segmentos retos e somamos os comprimentos.
- **Régua (YS, *yardstick*).** Caminhamos pela costa com uma régua de comprimento fixo
  $d$, como fez Lewis Fry Richardson com mapas (publicado em 1961).

Os dois comprimentos crescem quando a resolução fica mais fina. Para muitas costas
esse crescimento segue uma lei de potência, cujo expoente define uma *dimensão
fractal* $D$.

**Como usar.** Rode as células em ordem (*Ambiente de execução → Executar tudo* no
Colab); o notebook inteiro leva cerca de um minuto. As perguntas marcadas com
**Preveja** devem ser respondidas antes de rodar a célula seguinte; clique em
*Resposta* para comparar.

**Simplificações.** A Terra é tratada como plana, uma ótima aproximação para costas de
até algumas centenas de quilômetros (veja o Exercício 2), e a linha costeira é a
poligonal que passa pelos seus pontos (interpolação linear). O código completo da
análise, na esfera e com mais opções, está no
[repositório](https://github.com/fsaliby/Coastline_Length_Sampling).

## 1. Preparação
Bibliotecas, algumas constantes e uma semente aleatória fixa, para que cada execução
dê os mesmos números.

In [ ]:
import io
import urllib.request
import xml.etree.ElementTree as ET

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

plt.rcParams.update({"figure.figsize": (7.5, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})
BLUE, ORANGE, GREEN, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"

R_EARTH = 6371.0                          # raio da Terra, km
REPO = "https://raw.githubusercontent.com/fsaliby/Coastline_Length_Sampling/main/"
rng = np.random.default_rng(2026)         # semente fixa


def plain(axis):
    # eixo log com 20, 30 em vez de 2×10¹, 3×10¹
    axis.set_major_formatter(FormatStrFormatter("%g"))
    axis.set_minor_formatter(FormatStrFormatter("%g"))

## 2. A linha costeira

Escolha os dados com `SOURCE`:

- `"buzios"` (padrão): a península de Búzios, RJ, 38 km;
- `"britain"`: a costa oeste da Grã-Bretanha, de Land's End a Cape Wrath (base de
  dados GSHHG, 5470 km; leva uns 10 s a mais);
- `"upload"`: o seu próprio arquivo, que pode ser um KML com uma única linha (por
  exemplo desenhada no Google My Maps e exportada como KML) ou um arquivo de texto com
  duas colunas em graus. Ajuste `UPLOAD_FORMAT` para `"kml"`, `"latlon"` ou `"lonlat"`.

Os pontos são então projetados num plano (*Terra plana*), em torno da latitude média
$\bar\varphi$:
$$x = R\,(\lambda-\lambda_0)\cos\bar\varphi, \qquad y = R\,(\varphi-\varphi_0),$$
com latitude $\varphi$ e longitude $\lambda$ em radianos e $R = 6371$ km.

In [ ]:
SOURCE = "buzios"          # "buzios", "britain" ou "upload"
UPLOAD_FORMAT = "kml"      # só para "upload": "kml", "latlon" ou "lonlat"

DATASETS = {"buzios": ("Buzios_Peninsula.kml", "kml"),
            "britain": ("britain_west_gshhg.txt", "latlon")}


def load_text(source, fmt):
    # Texto do arquivo da costa: enviado, cópia local ou baixado do GitHub.
    if source == "upload":
        from google.colab import files        # só funciona no Colab
        name, data = next(iter(files.upload().items()))
        return data.decode("utf-8"), fmt, name
    name, fmt = DATASETS[source]
    try:
        with open(name, encoding="utf-8") as f:
            return f.read(), fmt, name
    except FileNotFoundError:
        return urllib.request.urlopen(REPO + name).read().decode("utf-8"), fmt, name


def parse_coastline(text, fmt):
    # (lat, lon) em graus, de uma linha KML ou de uma tabela com duas colunas.
    if fmt == "kml":
        root = ET.fromstring(text)
        lines = [c.text for el in root.iter() if el.tag.endswith(("LineString", "LinearRing"))
                 for c in el.iter() if c.tag.endswith("coordinates")]
        lon, lat = np.array([[float(v) for v in t.split(",")[:2]] for t in lines[0].split()]).T
    else:
        cols = np.loadtxt(io.StringIO(text.replace(",", " ")), comments="#")
        lat, lon = (cols[:, 0], cols[:, 1]) if fmt == "latlon" else (cols[:, 1], cols[:, 0])
    return lat, lon


def to_plane(lat, lon):
    # Projeção de Terra plana em torno da latitude média, em km. Pontos repetidos são removidos.
    lat0 = np.radians(np.mean(lat))
    x = R_EARTH * np.radians(lon - lon[0]) * np.cos(lat0)
    y = R_EARTH * np.radians(lat - lat[0])
    xy = np.column_stack([x, y])
    keep = np.r_[True, np.any(np.diff(xy, axis=0) != 0.0, axis=1)]
    return xy[keep]


def arc_length(xy):
    # Distância ao longo da poligonal, do primeiro ponto até cada vértice.
    return np.r_[0.0, np.cumsum(np.hypot(*np.diff(xy, axis=0).T))]


def points_at(xy, s_knots, s):
    # Pontos da poligonal nas distâncias s ao longo dela (qualquer formato de array).
    return np.stack([np.interp(s, s_knots, xy[:, 0]), np.interp(s, s_knots, xy[:, 1])], axis=-1)


def path_length(P):
    # Soma dos segmentos retos entre pontos consecutivos (último eixo = x, y).
    return np.hypot(*np.moveaxis(np.diff(P, axis=-2), -1, 0)).sum(axis=-1)


text, fmt, name = load_text(SOURCE, UPLOAD_FORMAT)
lat, lon = parse_coastline(text, fmt)
xy = to_plane(lat, lon)
s_knots = arc_length(xy)
L_ref = s_knots[-1]
seg = np.diff(s_knots)

print(f"{name}: {len(xy)} pontos, comprimento da poligonal L_ref = {L_ref:.3f} km")
print(f"distância entre as duas pontas: {np.hypot(*(xy[-1] - xy[0])):.2f} km;"
      f" espaçamento mediano dos pontos: {1000 * np.median(seg):.0f} m")

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot(xy[:, 0], xy[:, 1], color=GREY, lw=1)
ax.plot(*xy[0], "o", color=BLUE, label="início")
ax.plot(*xy[-1], "s", color=ORANGE, label="fim")
ax.set_aspect("equal")
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title(name)
ax.legend()
plt.show()

## 3. Uma medição com pontos aleatórios

As duas pontas da costa são mantidas, e $n_p - 2$ pontos são sorteados uniformemente
ao longo do seu comprimento. Ligando os $n_p$ pontos em ordem, obtemos uma poligonal
de $n_p - 1$ segmentos retos, cujo comprimento total $L$ é a medição.

**Preveja.** O comprimento medido $L$ será maior ou menor que $L_\text{ref}$, o
comprimento da poligonal da costa? Ele pode ser igual?

In [ ]:
def random_points(xy, n_p, n_real, rng):
    # n_real medições com n_p pontos cada: as duas pontas mais n_p - 2 pontos aleatórios.
    s_k = arc_length(xy)
    s = np.sort(rng.random((n_real, n_p - 2)) * s_k[-1], axis=1)
    s = np.column_stack([np.zeros(n_real), s, np.full(n_real, s_k[-1])])
    return points_at(xy, s_k, s)              # formato (n_real, n_p, 2)


P = random_points(xy, 10, 1, rng)[0]
L = path_length(P)
print(f"n_p = 10:  L = {L:.3f} km,  L / L_ref = {L / L_ref:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot(xy[:, 0], xy[:, 1], color=GREY, lw=1, label="costa")
ax.plot(P[:, 0], P[:, 1], "-o", color=BLUE, ms=5, label=f"medição, n_p = 10")
ax.set_aspect("equal")
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.legend()
plt.show()

<details><summary><b>Resposta</b></summary>

Sempre menor (ou igual). Cada segmento reto é o caminho mais curto entre seus dois
pontos, então nunca é mais longo que o trecho de costa que ele substitui (desigualdade
triangular). A igualdade exigiria que todo trecho de costa entre dois pontos
consecutivos fosse reto, o que nunca acontece numa costa real.
</details>

## 4. Muitas medições: a distribuição amostral

Outros pontos aleatórios dão outro comprimento: $L$ é uma variável aleatória. A sua
distribuição $P(L\,|\,n_p)$ é a **distribuição amostral** do estimador. A sua largura é
a variabilidade do *método*, não um erro nos dados da costa.

**Preveja.** Quando $n_p$ passa de 10 para 50, a largura da distribuição de $L$
aumenta ou diminui? E a média?

In [ ]:
N_REAL = 2000                             # medições por n_p

fig, ax = plt.subplots()
for n_p, colour in ((10, BLUE), (50, ORANGE)):
    L = path_length(random_points(xy, n_p, N_REAL, rng))
    skew = np.mean((L - L.mean()) ** 3) / L.std() ** 3
    print(f"n_p = {n_p:3d}:  média = {L.mean():8.3f} km,  desvio-padrão = {L.std(ddof=1):6.3f} km,"
          f"  relativo = {L.std(ddof=1) / L.mean():.3f},  assimetria = {skew:+.2f}")
    ax.hist(L, bins=50, density=True, alpha=0.6, color=colour, label=f"n_p = {n_p}")
ax.axvline(L_ref, color=GREY, ls="--", label="L_ref")
ax.set_xlabel("comprimento medido L (km)")
ax.set_ylabel("densidade de probabilidade")
ax.legend()
plt.show()

<details><summary><b>Resposta</b></summary>

A média cresce em direção a $L_\text{ref}$ (mais pontos acompanham mais detalhes da
costa) e a largura *relativa* diminui. Cada medição faz a média sobre mais segmentos.
</details>

## 5. Variando a resolução: a varredura aninhada

Agora medimos em muitos valores de $n_p$. Numa varredura **aninhada** (*nested*), cada
realização sorteia seus pontos aleatórios *uma única vez*, e a medição com $n_p$
pontos usa os $n_p - 2$ primeiros. É como um agrimensor que vai acrescentando pontos ao
mesmo mapa: cada $P(L\,|\,n_p)$ é igual à obtida com pontos novos, mas os valores de
uma mesma realização em diferentes $n_p$ são correlacionados.

**Por que uma lei de potência?** Richardson observou que o comprimento medido com uma
régua $\ell$ se comporta como $L \propto \ell^{\,1-D}$. Com $N = n_p - 1$ segmentos de
tamanho típico $\ell \approx L/N$, isso se torna
$$L = L_0\,(n_p - 1)^{\beta}, \qquad \beta = 1 - \frac{1}{D}, \qquad D = \frac{1}{1-\beta}.$$
Uma reta tem $D = 1$ ($\beta = 0$: o comprimento não muda); uma curva tão sinuosa que
preenchesse o plano teria $D = 2$.

In [ ]:
def rp_nested(xy, n_values, n_real, rng):
    # Comprimentos, formato (n_real, len(n_values)). Cada n_p usa os n_p - 2 primeiros pontos do mesmo sorteio.
    s_k = arc_length(xy)
    U = rng.random((n_real, max(n_values) - 2)) * s_k[-1]   # um sorteio por realização
    out = np.empty((n_real, len(n_values)))
    for j, n_p in enumerate(n_values):
        s = np.sort(U[:, :n_p - 2], axis=1)
        s = np.column_stack([np.zeros(n_real), s, np.full(n_real, s_k[-1])])
        out[:, j] = path_length(points_at(xy, s_k, s))
    return out


N_VALUES = np.unique(np.rint(np.geomspace(5, 500, 40)).astype(int))
L_rp = rp_nested(xy, N_VALUES, N_REAL, rng)
L_mean = L_rp.mean(axis=0)
lo, hi = np.percentile(L_rp, [16, 84], axis=0)

fig, ax = plt.subplots()
ax.fill_between(N_VALUES - 1, lo, hi, color=BLUE, alpha=0.25, lw=0, label="16-84% das medições")
ax.plot(N_VALUES - 1, L_mean, "o-", color=BLUE, ms=3, label="média <L>")
ax.axhline(L_ref, color=GREY, ls="--", label="L_ref")
ax.set_xscale("log")
ax.set_yscale("log")
plain(ax.yaxis)
ax.set_xlabel("número de segmentos n_p - 1")
ax.set_ylabel("L (km)")
ax.legend()
plt.show()

## 6. Ajustando a lei de potência: qual é a incerteza de $D$?

Tomando logaritmos, $\ln L = \ln L_0 + \beta \ln(n_p - 1)$ é uma reta, ajustada por
mínimos quadrados dentro de uma **janela** de valores de $n_p$ (a escolha é discutida
na Seção 7). A incerteza de $D$ pode ser estimada de duas maneiras:

1. **Ajustar a curva média** $\langle L\rangle$ uma vez e tomar o erro-padrão de
   $\beta$ da fórmula usual de mínimos quadrados; então $\sigma_D = D^2\sigma_\beta$
   (pois $dD/d\beta = D^2$).
2. **Monte Carlo:** ajustar cada realização separadamente e olhar a distribuição dos
   valores de $D$ obtidos.

**Preveja.** Qual das duas vai dar a maior incerteza?

In [ ]:
WINDOWS = {"buzios": (10, 50), "britain": (20, 200)}
N_MIN, N_MAX = WINDOWS.get(SOURCE, (10, 50))    # janela de ajuste em n_p (mude para os seus dados)


def fit_D(x, L):
    # Mínimos quadrados em ln L = ln L0 + beta ln x para cada linha de L. Retorna beta e D.
    beta = np.polyfit(np.log(x), np.log(np.atleast_2d(L)).T, 1)[0]
    return beta, 1.0 / (1.0 - beta)


win = (N_VALUES >= N_MIN) & (N_VALUES <= N_MAX)
x = N_VALUES[win] - 1.0

# 1. curva média, erro-padrão da fórmula de mínimos quadrados
(beta_m, _), cov = np.polyfit(np.log(x), np.log(L_mean[win]), 1, cov=True)
D_m = 1.0 / (1.0 - beta_m)
sigma_D = D_m ** 2 * np.sqrt(cov[0, 0])

# 2. Monte Carlo: um ajuste por realização
_, D_rp = fit_D(x, L_rp[:, win])
q16, q50, q84 = np.percentile(D_rp, [16, 50, 84])

print(f"janela: n_p = {N_MIN}..{N_MAX}  ({win.sum()} valores)")
print(f"1. curva média:  D = {D_m:.4f} ± {sigma_D:.4f}")
print(f"2. Monte Carlo:     D = {q50:.4f}  (16-84%: {q16:.4f} .. {q84:.4f}),  meia-largura {0.5 * (q84 - q16):.4f}")

C = np.corrcoef(L_rp[:, win].T)
r = np.mean(np.diag(C, 1))
print(f"correlação média entre valores vizinhos de n_p na mesma realização: {r:.2f}")

fig, ax = plt.subplots()
ax.hist(D_rp, bins=60, density=True, color=BLUE, alpha=0.7, label="Monte Carlo")
ax.axvspan(D_m - sigma_D, D_m + sigma_D, color=ORANGE, alpha=0.5, label="curva média ± erro-padrão")
ax.set_xlabel("D")
ax.set_ylabel("densidade de probabilidade")
ax.legend()
plt.show()

<details><summary><b>Resposta</b></summary>

A largura do Monte Carlo é muito maior. A fórmula de mínimos quadrados supõe que os
pontos se espalham de forma *independente* em torno da reta. Aqui os pontos da curva
média são médias de milhares de medições, quase sem espalhamento, e pontos vizinhos de
uma mesma realização são fortemente correlacionados (cerca de 0,9, impresso acima):
quando uma medição cai acima da tendência, as vizinhas em $n_p$ também caem. A fórmula então
mede apenas o quanto a curva média é bem descrita por uma reta.

Os dois números respondem a perguntas diferentes. A largura do Monte Carlo diz o
quanto $D$ mudaria se a medição inteira fosse repetida com outros pontos aleatórios; a
incerteza da sua mediana diminui como $1/\sqrt{N_\text{real}}$ (Exercício 3). Nenhum
dos dois inclui a escolha da janela, que é o assunto da próxima seção.
</details>

## 7. Onde ajustar? Dimensão local e a janela de escala

Uma lei de potência é uma reta num gráfico log-log. A sua inclinação *local*,
$\beta_\text{loc} = d\ln\langle L\rangle / d\ln(n_p - 1)$, dá uma dimensão local
$D_\text{loc} = 1/(1-\beta_\text{loc})$ em cada resolução. Ela é desenhada em função
da **régua efetiva** $\ell = \langle L\rangle/(n_p - 1)$, o comprimento médio de um
segmento.

Uma curva autossimilar mantém $D_\text{loc}$ constante numa faixa de escalas: um
**plateau**, onde a janela de ajuste deve ficar. Para réguas próximas do espaçamento
dos pontos dos dados, a poligonal parece reta e $D_\text{loc} \to 1$; para réguas
comparáveis ao tamanho da costa, a lei de potência também não vale.

In [ ]:
def local_dimension(n_values, L_mean):
    beta = np.gradient(np.log(L_mean), np.log(np.asarray(n_values) - 1.0))
    return 1.0 / (1.0 - beta)


ell = L_mean / (N_VALUES - 1)                    # régua efetiva, km
D_loc = local_dimension(N_VALUES, L_mean)

fig, ax = plt.subplots()
ax.plot(ell, D_loc, "o-", color=BLUE, ms=3)
ax.axvspan(ell[win].min(), ell[win].max(), color=BLUE, alpha=0.12, label=f"janela de ajuste n_p = {N_MIN}..{N_MAX}")
ax.axhline(1.0, color=GREY, ls=":")
ax.set_xscale("log")
ax.set_xlabel("régua efetiva <L>/(n_p - 1)  (km)")
ax.set_ylabel("dimensão local D_loc")
ax.legend()
plt.show()

print("janela n_p       mediana D   16-84%")
for a, b in ((5, 20), (10, 50), (20, 100), (50, 200), (100, 500)):
    w = (N_VALUES >= a) & (N_VALUES <= b)
    _, D = fit_D(N_VALUES[w] - 1.0, L_rp[:, w])
    print(f"{a:4d} .. {b:<4d}      {np.median(D):.4f}     {np.percentile(D, 16):.4f} .. {np.percentile(D, 84):.4f}")

A largura dentro de uma janela é a incerteza **estatística**; a mudança da mediana de
uma janela razoável para outra é uma incerteza **sistemática**, muitas vezes a maior
das duas. Um resultado deve informar as duas e mostrar o plateau que justifica a
janela.

## 8. A régua

Um levantamento começa num ponto aleatório da costa e caminha nos dois sentidos. A
partir de cada estação, a próxima é o **primeiro** ponto mais adiante na costa cuja
distância em linha reta é exatamente $d$. Numa poligonal, esse ponto é encontrado de
forma exata:

1. percorremos os vértices à frente e paramos no primeiro, $\mathbf V$, que fica fora
   do círculo de raio $d$ em torno da estação $\mathbf P$;
2. o cruzamento está no segmento que termina em $\mathbf V$, a partir da sua
   extremidade interna $\mathbf E$ (dentro do círculo). Escrevendo
   $\mathbf X = \mathbf E + w(\mathbf V-\mathbf E)$ com $0 < w \le 1$, a condição
   $|\mathbf X - \mathbf P| = d$ é uma equação do segundo grau,
   $$a w^2 + 2 b w + c = 0,\quad a = |\mathbf V-\mathbf E|^2,\quad
   b = (\mathbf E-\mathbf P)\cdot(\mathbf V-\mathbf E),\quad c = |\mathbf E-\mathbf P|^2 - d^2,$$
   cuja raiz é $w = \left(-b + \sqrt{b^2 - ac}\right)/a$.

Em cada ponta da linha sobra um pedaço menor que $d$. Se ele for menor que $d/2$, a
última estação é removida e um segmento vai direto até a ponta; senão, o pedaço é
mantido como mais um segmento, mais curto. Nos dois casos $n_p - 1 \approx L/d$, e
$n_p$ conta todas as estações, incluindo as duas pontas, como nos pontos aleatórios.

Todas as réguas usam os mesmos pontos de partida aleatórios (*sementes comuns*), então
cada levantamento é uma realização em todas as réguas. As réguas são escolhidas para
corresponder à janela do RP: da régua efetiva em $n_p = $ `N_MAX` até a de
$n_p = $ `N_MIN`.

In [ ]:
def walk(xy, s_k, s0, d):
    # Estações de uma caminhada com régua d, da distância s0 ao longo da linha até o fim.
    i = min(np.searchsorted(s_k, s0, side="right") - 1, len(xy) - 2)   # segmento da partida
    P = points_at(xy, s_k, s0)
    stations = [P]
    while True:
        # 1. primeiro vértice à frente fora do círculo de raio d (busca em blocos de 64)
        k, start = None, i + 1
        while start < len(xy):
            stop = min(start + 64, len(xy))
            outside = np.flatnonzero(np.hypot(*(xy[start:stop] - P).T) >= d)
            if outside.size:
                k = start + outside[0]
                break
            start = stop
        if k is None:                         # o fim da linha está a menos de d
            return np.array(stations)
        # 2. cruzamento no segmento que termina no vértice k
        E = P if k - 1 == i else xy[k - 1]
        V = xy[k]
        a = (V - E) @ (V - E)
        b = (E - P) @ (V - E)
        c = (E - P) @ (E - P) - d * d
        P = E + (-b + np.sqrt(b * b - a * c)) / a * (V - E)
        stations.append(P)
        i = k - 1


def yardstick(xy, d, s0):
    # Um levantamento a partir da semente s0, nos dois sentidos. Retorna (L, n_p, estações).
    s_k = arc_length(xy)
    rev = xy[::-1]
    forward = walk(xy, s_k, s0, d)
    backward = walk(rev, arc_length(rev), s_k[-1] - s0, d)
    # pedaço da ponta menor que d/2: remove a última estação, um segmento vai até a ponta
    if len(forward) > 1 and np.hypot(*(xy[-1] - forward[-1])) < d / 2:
        forward = forward[:-1]
    if len(backward) > 1 and np.hypot(*(xy[0] - backward[-1])) < d / 2:
        backward = backward[:-1]
    path = np.vstack([xy[:1], backward[::-1], forward[1:], xy[-1:]])
    return path_length(path), len(path), path


ell_win = ell[win]
RULERS = np.geomspace(ell_win.min(), ell_win.max(), 12)
N_SURVEYS = 300 if len(xy) < 5000 else 100     # menos levantamentos para costas longas

# um levantamento, desenhado no mapa
L1, n1, path = yardstick(xy, RULERS[-1], 0.5 * L_ref)
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot(xy[:, 0], xy[:, 1], color=GREY, lw=1, label="costa")
ax.plot(path[:, 0], path[:, 1], "-o", color=ORANGE, ms=4, label=f"régua d = {RULERS[-1]:.3g} km: n_p = {n1}, L = {L1:.2f} km")
ax.set_aspect("equal")
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.legend()
plt.show()

In [ ]:
s0 = rng.random(N_SURVEYS) * L_ref               # sementes comuns para todas as réguas
L_ys = np.empty((N_SURVEYS, RULERS.size))
n_ys = np.empty((N_SURVEYS, RULERS.size), dtype=int)
for j, d in enumerate(RULERS):
    for k in range(N_SURVEYS):
        L_ys[k, j], n_ys[k, j], _ = yardstick(xy, d, s0[k])

# Richardson: ln L = const + (1 - D) ln d, um ajuste por levantamento
slope = np.polyfit(np.log(RULERS), np.log(L_ys).T, 1)[0]
D_ys_d = 1.0 - slope
# como nos pontos aleatórios: ln L contra ln(n_p - 1), um ajuste por levantamento
D_ys_n = np.array([fit_D(n_ys[k] - 1.0, L_ys[k])[1][0] for k in range(N_SURVEYS)])

for label, D in (("L contra d      ", D_ys_d), ("L contra n_p - 1", D_ys_n)):
    print(f"YS, {label}:  D = {np.median(D):.4f}  (16-84%: {np.percentile(D, 16):.4f} .. {np.percentile(D, 84):.4f})")

fig, ax = plt.subplots()
ax.fill_between(RULERS, *np.percentile(L_ys, [16, 84], axis=0), color=ORANGE, alpha=0.25, lw=0, label="16-84%")
ax.plot(RULERS, L_ys.mean(axis=0), "s-", color=ORANGE, ms=4, label="média <L>")
ax.set_xscale("log")
ax.set_yscale("log")
plain(ax.xaxis)
plain(ax.yaxis)
ax.set_xlabel("régua d (km)")
ax.set_ylabel("L (km)")
ax.set_title("Gráfico de Richardson")
ax.legend()
plt.show()

## 9. Dois estimadores, uma curva

Os dois estimadores são construídos de forma diferente: os pontos aleatórios fixam o
*número* de segmentos e deixam seus comprimentos variar; a régua fixa o *comprimento*
dos segmentos e deixa o seu número variar. Desenhados em função do comprimento de um
segmento, a régua efetiva $\ell$ nos pontos aleatórios e $d$ na régua, os dois
descrevem a mesma costa.

**Preveja.** As duas curvas vão coincidir?

In [ ]:
fig, ax = plt.subplots()
ax.plot(ell, L_mean, "o-", color=BLUE, ms=3, label="pontos aleatórios: <L> contra <L>/(n_p - 1)")
ax.plot(RULERS, L_ys.mean(axis=0), "s-", color=ORANGE, ms=4, label="régua: <L> contra d")
ax.axvspan(ell_win.min(), ell_win.max(), color=GREY, alpha=0.1)
ax.set_xscale("log")
ax.set_yscale("log")
plain(ax.yaxis)
ax.set_xlabel("comprimento de um segmento (km)")
ax.set_ylabel("<L> (km)")
ax.legend()
plt.show()

print(f"pontos aleatórios, n_p = {N_MIN}..{N_MAX}:   D = {np.median(D_rp):.4f}  (16-84%: {np.percentile(D_rp, 16):.4f} .. {np.percentile(D_rp, 84):.4f})")
print(f"régua, d = {RULERS[0]:.3g}..{RULERS[-1]:.3g} km:  D = {np.median(D_ys_d):.4f}  (16-84%: {np.percentile(D_ys_d, 16):.4f} .. {np.percentile(D_ys_d, 84):.4f})")

<details><summary><b>Resposta</b></summary>

Quase, mas não exatamente: um segmento dos pontos aleatórios e uma régua com o mesmo
comprimento médio não cortam as mesmas curvas, porque os segmentos aleatórios variam
de tamanho. O que importa é que os dois dão uma *inclinação* parecida, e portanto um
$D$ parecido. A régua costuma dar uma distribuição de $D$ mais estreita: todos os seus
segmentos têm o mesmo comprimento, então cada levantamento tem menos aleatoriedade.
</details>

## 10. Validação: o método encontra uma dimensão conhecida?

Antes de confiar numa medição, teste o método onde a resposta é conhecida. Duas curvas
sintéticas do mesmo tamanho da costa de Búzios (7 km entre as pontas, cerca de
38–39 km de comprimento):

- uma **curva de Koch**: cada segmento é substituído por quatro segmentos de um terço
  do seu comprimento, seis vezes seguidas, então $D = \ln 4 / \ln 3 \approx 1{,}262$
  exatamente;
- uma **senoide suave**, com três períodos: uma curva lisa, então $D = 1$.

As duas são medidas com pontos aleatórios e ajustadas na mesma janela $n_p$ = 10..50.

**Preveja.** O que o ajuste vai dar para a senoide?

In [ ]:
def koch(level, base=7.0):
    # Curva de Koch de (0, 0) a (base, 0): cada segmento vira quatro com um terço do seu comprimento.
    P = np.array([[0.0, 0.0], [base, 0.0]])
    turn = np.array([[0.5, -np.sqrt(3) / 2], [np.sqrt(3) / 2, 0.5]])   # rotação de 60°
    for _ in range(level):
        A, B = P[:-1], P[1:]
        third = (B - A) / 3
        P = np.vstack([np.stack([A, A + third, A + third + third @ turn.T, A + 2 * third], axis=1).reshape(-1, 2), P[-1:]])
    return P


t = np.linspace(0.0, 1.0, 4001)
TEST_CURVES = {
    "Koch (D = 1.262)": koch(6),
    "senoide (D = 1)": np.column_stack([7.0 * t, 3.1 * np.sin(6 * np.pi * t)]),
}
w_test = (N_VALUES >= 10) & (N_VALUES <= 50)

fig, ax = plt.subplots()
for (label, curve), colour in zip(TEST_CURVES.items(), (ORANGE, GREEN)):
    Lc = rp_nested(curve, N_VALUES, N_REAL, rng)
    _, Dc = fit_D(N_VALUES[w_test] - 1.0, Lc[:, w_test])
    mean_c = Lc.mean(axis=0)
    print(f"{label:22s} comprimento {arc_length(curve)[-1]:5.1f} km:  ajustado D = {np.median(Dc):.3f}  (16-84%: {np.percentile(Dc, 16):.3f} .. {np.percentile(Dc, 84):.3f})")
    ax.plot(mean_c / (N_VALUES - 1), local_dimension(N_VALUES, mean_c), "o-", ms=3, color=colour, label=label)
ax.plot(ell, D_loc, "o-", ms=3, color=BLUE, label=f"{name}")
ax.axhline(np.log(4) / np.log(3), color=ORANGE, ls=":")
ax.axhline(1.0, color=GREEN, ls=":")
ax.set_xscale("log")
ax.set_ylim(0.95, 1.8)
ax.set_xlabel("régua efetiva <L>/(n_p - 1)  (km)")
ax.set_ylabel("dimensão local D_loc")
ax.legend()
plt.show()

<details><summary><b>Resposta</b></summary>

A curva de Koch dá um valor próximo de 1,262, um pouco abaixo: com réguas tão
grandes, os pontos aleatórios subestimam em alguns centésimos, e o gráfico da dimensão
local mostra o plateau do Koch em 1,26 nas réguas menores. A senoide suave, cuja
dimensão verdadeira é 1, dá um valor bem **acima** de 1 nesta janela, porque as suas
ondulações têm mais ou menos o tamanho da janela. Um $D > 1$ ajustado numa janela,
sozinho, *não* é evidência de estrutura fractal. O gráfico da dimensão local separa os
dois casos: a curva de Koch tem um plateau longo, enquanto a senoide tende a 1 nas
réguas pequenas. Por isso o plateau deve sempre acompanhar o resultado.
</details>

## 11. Sua vez

1. **A sua costa.** Desenhe uma linha costeira no Google My Maps, exporte como KML,
   coloque `SOURCE = "upload"` e rode tudo de novo. Escolha `N_MIN` e `N_MAX` a partir
   do plateau da Seção 7.
2. **A Terra é plana o suficiente?** Escreva a distância de grande círculo (haversine)
   $$\ell = 2R\arcsin\sqrt{\sin^2\tfrac{\Delta\varphi}{2} + \cos\varphi_1\cos\varphi_2\sin^2\tfrac{\Delta\lambda}{2}},$$
   use-a no lugar de `path_length` com `SOURCE = "britain"` (950 km de sul a norte) e
   compare $L$ e $D$ com os valores da Terra plana.
3. **Mais realizações.** Dobre `N_REAL`. O que muda mais: a mediana de $D$ ou a
   largura da sua distribuição? Por quê?
4. **Incerteza sistemática.** Encontre a janela mais larga em que a mediana de $D$
   muda menos de 0,01. Informe $D$ com as incertezas estatística e sistemática.
5. **Independente em vez de aninhado.** Mude `rp_nested` para que cada $n_p$ sorteie
   pontos novos. A distribuição de $D$ fica mais larga ou mais estreita? Explique com a
   correlação da Seção 6.
6. **A régua na curva de Koch.** Rode a Seção 8 com `xy = koch(6)`. Como $n_p$ muda
   com $d$? (Olhe os valores: a curva de Koch é autossimilar com fator 3.)

## 12. Para ir além

- O código completo da análise (na esfera, com interpolação por splines, ilhas
  fechadas, 10⁵–10⁶ realizações e várias opções de ajuste) e os conjuntos de dados de
  exemplo estão no repositório [fsaliby/Coastline_Length_Sampling](https://github.com/fsaliby/Coastline_Length_Sampling).
- L. F. Richardson, "The problem of contiguity: an appendix to *Statistics of deadly
  quarrels*", *General Systems Yearbook* **6**, 139–187 (1961).
- B. Mandelbrot, "How long is the coast of Britain? Statistical self-similarity and
  fractional dimension", *Science* **156**, 636–638 (1967).
- P. Wessel e W. H. F. Smith, "A global, self-consistent, hierarchical,
  high-resolution shoreline database", *J. Geophys. Res.* **101**(B4), 8741–8743
  (1996): a fonte dos dados da Grã-Bretanha (GSHHG).